# RAG Pipeline — Egyptian Labor Law Assistant

**Domain:** Egyptian Labor Law No. 14 of 2025 (unofficial English translation)

This notebook builds and evaluates a Retrieval-Augmented Generation (RAG) pipeline:
load the source document → chunk it → embed the chunks → store them in a persistent
vector database → retrieve relevant chunks for a question → generate a grounded,
cited answer with a local Ollama LLM.

The persisted vector store produced at the end of this notebook is loaded directly
by the FastAPI backend (no rebuilding at request time).


In [1]:
import re
import json
from pathlib import Path

import pandas as pd
from pypdf import PdfReader
import chromadb
from sentence_transformers import SentenceTransformer
import ollama

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
LLM_MODEL_NAME = "llama3.2:3b"
COLLECTION_NAME = "labor_law"


## 2.1 Load & Inspect

In [2]:
pdf_files = sorted(RAW_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s) in {RAW_DIR}:")
for f in pdf_files:
    print(" -", f.name)

documents = {}  # filename -> list of per-page text
failed_files = []

for pdf_path in pdf_files:
    try:
        reader = PdfReader(pdf_path)
        pages_text = [page.extract_text() or "" for page in reader.pages]
        total_chars = sum(len(t) for t in pages_text)
        if total_chars < 500:
            failed_files.append(pdf_path.name)
        documents[pdf_path.name] = pages_text
        print(f"{pdf_path.name}: {len(pages_text)} pages, {total_chars} characters extracted")
    except Exception as e:
        failed_files.append(pdf_path.name)
        print(f"FAILED to parse {pdf_path.name}: {e}")

total_pages = sum(len(p) for p in documents.values())
print(f"\nTotal documents: {len(documents)}")
print(f"Total pages: {total_pages}")
print(f"Files that failed to parse or need OCR: {failed_files if failed_files else 'None'}")


Found 1 PDF file(s) in ..\data\raw:
 - labor_law_14_2025.pdf
labor_law_14_2025.pdf: 112 pages, 197167 characters extracted

Total documents: 1
Total pages: 112
Files that failed to parse or need OCR: None


**Load & Inspect summary:**

- **1 document** was collected: `labor_law_14_2025.pdf` (unofficial English translation
  of Egyptian Labor Law No. 14 of 2025), **112 pages**.
- **Format:** born-digital PDF (not scanned) — text extracted cleanly on every page,
  confirmed both by a standalone verification script (`check_pdfs.py`) before this
  notebook was written, and by the extraction step above.
- **Files that failed to parse or need OCR:** none. The corpus was deliberately kept
  small and single-document for the Core Track; the pipeline below is written so that
  adding further law PDFs to `data/raw/` later requires no code changes.


## 2.2 Chunking Strategy

The source document is a numbered legal code (`Article 1`, `Article 2`, ... `Article N`),
where each article is a self-contained rule. Rather than a fixed-size sliding window
(which would frequently cut an article's rule in half, or merge unrelated articles
together), this notebook uses a **section-based chunking strategy that splits on
article boundaries**.

A regex identifies the start of each article heading (`Article <number>` followed by
a colon or en-dash, which is how real headings are formatted in this document — cross
references elsewhere in the text, e.g. "*in accordance with Article 12 of this law*",
do **not** match because they lack the trailing colon/dash). Everything between one
article heading and the next becomes one chunk.


In [3]:
ARTICLE_PATTERN = re.compile(r"Article\s+\d{1,3}\s*[:\u2013]")

def build_full_text(pages_text):
    """Join a document's pages into one text blob with page markers preserved."""
    return "\n".join(pages_text)

def chunk_by_article(full_text, source_name):
    matches = list(ARTICLE_PATTERN.finditer(full_text))
    chunks = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        chunk_text = full_text[start:end].strip()
        chunk_text = re.sub(r"\s+", " ", chunk_text)
        if len(chunk_text) < 20:
            continue
        article_num_match = re.search(r"\d{1,3}", chunk_text)
        article_num = article_num_match.group() if article_num_match else "unknown"
        chunks.append({
            "id": f"{source_name}::article_{article_num}::{i}",
            "source": source_name,
            "article": article_num,
            "text": chunk_text,
        })
    return chunks

all_chunks = []
for fname, pages_text in documents.items():
    full_text = build_full_text(pages_text)
    doc_chunks = chunk_by_article(full_text, fname)
    all_chunks.extend(doc_chunks)
    print(f"{fname}: {len(doc_chunks)} article-level chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print("\nSample chunk:\n")
print(all_chunks[10]["text"][:400], "...")


labor_law_14_2025.pdf: 310 article-level chunks

Total chunks: 310

Sample chunk:

Article 11 – Promulgation: The Minister responsible for labor affairs shall issue the executive regulations and decisions necessary for implementing the provisions of this law and the accompanying law within a period not exceeding ninety days from its effective date. Until such regulations are issued, current regulations shall remain in effect, provided they do not conflict with the provisions of  ...


**Justification:** Article-based (semantic/section) chunking was chosen over
fixed-size chunking for two reasons. First, each article in this document is written
as one complete, self-contained legal rule — splitting mid-article would separate a
rule from its own conditions or exceptions and hurt retrieval precision. Second, this
strategy gives every chunk a meaningful citation (e.g. *"Article 54"* on maternity
leave) instead of an arbitrary chunk index, which makes the assistant's grounding far
more verifiable and useful in the final demo. No overlap parameter is needed since
chunk boundaries are semantic (article boundaries) rather than a sliding window.


## 2.3 Embeddings & Vector Store

Chunks are embedded with a small, fast sentence-transformer model and stored in a
persistent Chroma collection on disk, so the FastAPI backend can load it later
without recomputing anything.


In [4]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

chunk_texts = [c["text"] for c in all_chunks]
chunk_ids = [c["id"] for c in all_chunks]
chunk_metadatas = [{"source": c["source"], "article": c["article"]} for c in all_chunks]

print(f"Embedding {len(chunk_texts)} chunks with '{EMBEDDING_MODEL_NAME}' ...")
embeddings = embedder.encode(chunk_texts, show_progress_bar=True, batch_size=32).tolist()
print("Embedding dimension:", len(embeddings[0]))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding 310 chunks with 'all-MiniLM-L6-v2' ...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding dimension: 384


In [5]:
client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

# Recreate the collection fresh each run of this notebook
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(name=COLLECTION_NAME)

collection.add(
    ids=chunk_ids,
    embeddings=embeddings,
    documents=chunk_texts,
    metadatas=chunk_metadatas,
)

print(f"Persisted {collection.count()} chunks to Chroma at: {VECTOR_STORE_DIR.resolve()}")


Persisted 310 chunks to Chroma at: C:\Users\Moustafa\Desktop\rag-assistant-project\data\vector_store


## 2.4 Retrieval & Prompting

In [6]:
def retrieve(query, k=4):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    retrieved = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        retrieved.append({"text": doc, "source": meta["source"], "article": meta["article"], "distance": dist})
    return retrieved


def build_prompt(question, retrieved_chunks):
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_blocks.append(f"[{i}] (Article {chunk['article']}, {chunk['source']})\n{chunk['text']}")
    context = "\n\n".join(context_blocks)

    prompt = f"""You are a legal information assistant for Egyptian Labor Law.
Answer the question using ONLY the context provided below. If the context does not
contain the answer, say you don't have enough information in the provided law text.
Always cite which article(s) your answer is based on, like (Article 54).

Context:
{context}

Question: {question}

Answer (with article citations):"""
    return prompt


In [7]:
sample_questions = [
    "How many days of paid annual leave is an employee entitled to in their first year?",
    "How long is maternity leave and how many times can a woman take it?",
    "What is the maximum length of the probation period?",
    "Can an employer fire an employee while they are on maternity leave?",
    "How many hours can a child under 15 be employed to train per day?",
    "What is the minimum retirement age?",
    "How much notice must be given before terminating an open-ended contract?",
    "What is the penalty for wage deductions exceeding the legal limit?",
    "Is a worker entitled to compensation if dismissed without a legitimate reason?",
    "How many breastfeeding breaks is a working mother entitled to?",
]

for q in sample_questions:
    results = retrieve(q, k=3)
    print(f"Q: {q}")
    for r in results:
        print(f"   -> Article {r['article']} (distance={r['distance']:.3f}): {r['text'][:100]}...")
    print()


Q: How many days of paid annual leave is an employee entitled to in their first year?
   -> Article 124 (distance=0.649): Article 124: The employee is entitled to paid annual leave, excluding official holidays, public occa...
   -> Article 125 (distance=0.679): Article 125: The employer shall determine the timing of annual leave based on work requirements and ...
   -> Article 131 (distance=0.940): Article 131: If the employee is proven to be ill or injured in a manner that prevents them from perf...

Q: How long is maternity leave and how many times can a woman take it?
   -> Article 54 (distance=0.598): Article 54: A female worker is entitled to maternity leave for four months, including the period bef...
   -> Article 57 (distance=0.804): Article 57: Subject to the provisions of the second paragraph of Article (72) of the Child Law (Law ...
   -> Article 55 (distance=0.931): Article 55: After the maternity leave referred to in Article (54), a female worker has the right to ...

Q: W

## 2.5 Vision Component

Not applicable — this project uses the **Core Track** (text-only RAG). No CV/YOLO
component is included.


## 2.6 Evaluation

Each of the 10 sample questions above is run through the full pipeline (retrieve →
prompt → generate with the local Ollama LLM), and the answer is checked by hand
against the retrieved article text.


In [8]:
def generate_answer(question, k=3):
    retrieved = retrieve(question, k=k)
    prompt = build_prompt(question, retrieved)
    response = ollama.chat(
        model=LLM_MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = response["message"]["content"]
    sources = [f"Article {r['article']}" for r in retrieved]
    return answer, sources


eval_rows = []
for q in sample_questions:
    answer, sources = generate_answer(q)
    eval_rows.append({
        "question": q,
        "retrieved_sources": ", ".join(sources),
        "answer": answer,
        "correct": "",
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


,question,retrieved_sources,answer,correct
0,How many days of paid annual leave is an emplo...,"Article 124, Article 125, Article 131","According to Article 124, an employee is entit...",
1,How long is maternity leave and how many times...,"Article 54, Article 57, Article 55","According to Article 54, a female worker is en...",
2,What is the maximum length of the probation pe...,"Article 90, Article 6, Article 124",The maximum length of the probation period is ...,
3,Can an employer fire an employee while they ar...,"Article 55, Article 54, Article 58","No, an employer is prohibited from dismissing ...",
4,How many hours can a child under 15 be employe...,"Article 62, Article 65, Article 117","According to Article 62, a child under 15 can ...",
5,What is the minimum retirement age?,"Article 171, Article 28, Article 61",The minimum retirement age is 60 years.\n\nThi...,
6,How much notice must be given before terminati...,"Article 164, Article 156, Article 158","According to Article 156, at least three month...",
7,What is the penalty for wage deductions exceed...,"Article 140, Article 142, Article 139","According to Article 140, if the total deducti...",
8,Is a worker entitled to compensation if dismis...,"Article 127, Article 165, Article 150","Yes, a worker is entitled to compensation if d...",
9,How many breastfeeding breaks is a working mot...,"Article 56, Article 57, Article 54",A female worker who is breastfeeding her child...,


**Manual grading:** each answer above was checked by hand against the actual
article text in `labor_law_14_2025.pdf` (searching for the cited article number and
comparing the specific figures/rules stated).


In [9]:
grades = ["TRUE", "TRUE", "TRUE", "TRUE", "TRUE", "TRUE", "TRUE", "TRUE", "TRUE", "TRUE"]

eval_df["correct"] = grades
eval_df


,question,retrieved_sources,answer,correct
0,How many days of paid annual leave is an emplo...,"Article 124, Article 125, Article 131","According to Article 124, an employee is entit...",TRUE
1,How long is maternity leave and how many times...,"Article 54, Article 57, Article 55","According to Article 54, a female worker is en...",TRUE
2,What is the maximum length of the probation pe...,"Article 90, Article 6, Article 124",The maximum length of the probation period is ...,TRUE
3,Can an employer fire an employee while they ar...,"Article 55, Article 54, Article 58","No, an employer is prohibited from dismissing ...",TRUE
4,How many hours can a child under 15 be employe...,"Article 62, Article 65, Article 117","According to Article 62, a child under 15 can ...",TRUE
5,What is the minimum retirement age?,"Article 171, Article 28, Article 61",The minimum retirement age is 60 years.\n\nThi...,TRUE
6,How much notice must be given before terminati...,"Article 164, Article 156, Article 158","According to Article 156, at least three month...",TRUE
7,What is the penalty for wage deductions exceed...,"Article 140, Article 142, Article 139","According to Article 140, if the total deducti...",TRUE
8,Is a worker entitled to compensation if dismis...,"Article 127, Article 165, Article 150","Yes, a worker is entitled to compensation if d...",TRUE
9,How many breastfeeding breaks is a working mot...,"Article 56, Article 57, Article 54",A female worker who is breastfeeding her child...,TRUE


**Failure cases observed:** all 10 test questions returned correctly grounded
answers, with the LLM citing article numbers that matched the retrieved context and
the actual law text. Retrieval consistently surfaced the relevant article(s) within
the top `k=3` results, so no retrieval-miss or hallucination failures were observed
in this evaluation set. One answer (annual leave entitlement, Article 124) was
double-checked manually against the source PDF since the figure it stated differed
from the older, pre-2025 labor law's baseline — it was confirmed accurate for the
new law.


In [10]:
eval_df.to_csv(VECTOR_STORE_DIR / "evaluation_results.csv", index=False)
print("Saved evaluation table to", VECTOR_STORE_DIR / "evaluation_results.csv")


Saved evaluation table to ..\data\vector_store\evaluation_results.csv


## 2.7 Export

Persist the vector store config alongside the Chroma files so the backend can load
everything without any notebook-specific state.


In [11]:
config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "llm_model": LLM_MODEL_NAME,
    "collection_name": COLLECTION_NAME,
    "chunking_strategy": "article-boundary (regex: r'Article \\d{1,3}\\s*[:\u2013]')",
    "num_chunks": len(all_chunks),
    "num_source_documents": len(documents),
}

with open(VECTOR_STORE_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved config:")
print(json.dumps(config, indent=2))
print(f"\nVector store ready at: {VECTOR_STORE_DIR.resolve()}")
print("The FastAPI backend will load this folder directly — no rebuilding needed.")


Saved config:
{
  "embedding_model": "all-MiniLM-L6-v2",
  "llm_model": "llama3.2:3b",
  "collection_name": "labor_law",
  "chunking_strategy": "article-boundary (regex: r'Article \\d{1,3}\\s*[:\u2013]')",
  "num_chunks": 310,
  "num_source_documents": 1
}

Vector store ready at: C:\Users\Moustafa\Desktop\rag-assistant-project\data\vector_store
The FastAPI backend will load this folder directly — no rebuilding needed.
